In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
import os


In [ ]:
# Load the data
patient_data = pd.read_excel('Data/tcga-clinical patient metadata_COAD.xlsx')

In [ ]:
patient_data.head()

In [ ]:
patient_data.describe()

In [ ]:
patient_data.info()

In [ ]:
patient_data.shape

In [ ]:
# Load the resnet model
model = tf.keras.models.load_model('resnet_model_final')

In [ ]:
# Read in all images

# path to dataset directory
dataset_path = 'Data/ALL_IMAGES'  

# Create an ImageDataGenerator object
datagen = ImageDataGenerator(dtype='float32', preprocessing_function=preprocess_input)

# Set the batch size
batch_size = 32

# Load all images from the unified directory
images_dataset = datagen.flow_from_directory(
    directory=dataset_path,
    target_size=(224, 224),  # Resize all images to 224x224
    batch_size=batch_size,
    class_mode='binary',  # Use 'binary' for binary classification problems
    shuffle=False  # Set to False if you need consistent order for predictions
)

In [ ]:
# Extract filenames from the images_dataset
filenames = images_dataset.filenames

# Create a dictionary to count occurrences of each filename
filename_counts = {}

# Check for duplicates
duplicates = []
for filename in filenames:
    if filename in filename_counts:
        filename_counts[filename] += 1
    else:
        filename_counts[filename] = 1

# Collect filenames that appear more than once
duplicates = [filename for filename, count in filename_counts.items() if count > 1]

# Output the result
if duplicates:
    print("Duplicate filenames found:", duplicates)
else:
    print("No duplicate filenames found.")


## Preprocessing Patient Data

In [ ]:
# which columns that have all null values
patient_data.columns[patient_data.isnull().all()]

In [ ]:
# count of columns that have all null values
len(patient_data.columns[patient_data.isnull().all()])

In [ ]:
# drop columns that have all null values
patient_data = patient_data.dropna(axis=1, how='all')

In [ ]:
# print columns that have more than 1 null value and their count of null values
for col in patient_data.columns:
    if patient_data[col].isnull().sum() > 1:
        print(f'{col}: {patient_data[col].isnull().sum()}')


In [ ]:
# for the 'M' column print number of values with 'X'
print(f"Number of values with 'X': {patient_data['M'].value_counts()['X']}")

In [ ]:
# for M column replace X and null values with mode value
patient_data['M'] = patient_data['M'].replace('X', patient_data['M'].mode()[0])
patient_data['M'] = patient_data['M'].fillna(patient_data['M'].mode()[0])

In [ ]:
# print M column value counts
print(patient_data['M'].value_counts())

In [ ]:
# remove columns with more than 50% null values
patient_data = patient_data.dropna(thresh=0.5*patient_data.shape[0], axis=1)

In [ ]:
# print columns with null values and their count of null values
for col in patient_data.columns:
    if patient_data[col].isnull().sum() > 0:
        print(f'{col}: {patient_data[col].isnull().sum()}')

In [ ]:
# mode imputation for summarystage column
patient_data['summarystage'] = patient_data['summarystage'].fillna(patient_data['summarystage'].mode()[0])

In [ ]:
patient_data.head()

In [ ]:
# mode imputation for recurrence_status column
patient_data['recurrence_status'] = patient_data['recurrence_status'].fillna(patient_data['recurrence_status'].mode()[0])

In [ ]:
# mode imputation for vital_status column
patient_data['vital_status'] = patient_data['vital_status'].fillna(patient_data['vital_status'].mode()[0])

In [ ]:
# mode imputation for location column
patient_data['location'] = patient_data['location'].fillna(patient_data['location'].mode()[0])

In [ ]:
# mode imputation for summarylocation column
patient_data['summarylocation'] = patient_data['summarylocation'].fillna(patient_data['summarylocation'].mode()[0])

In [ ]:
# median imputation for lymphnodesremoved column
patient_data['lymphnodesremoved'] = patient_data['lymphnodesremoved'].fillna(patient_data['lymphnodesremoved'].median())

In [ ]:
# median imputation for lymphnodesinvaded column
patient_data['lymphnodesinvaded'] = patient_data['lymphnodesinvaded'].fillna(patient_data['lymphnodesinvaded'].median())

In [ ]:
# median imputation for stageall column
patient_data['stageall'] = patient_data['stageall'].fillna(patient_data['stageall'].median())

In [ ]:
# mode imputation for ethnicity column 
patient_data['ethnicity'] = patient_data['ethnicity'].fillna(patient_data['ethnicity'].mode()[0])

In [ ]:
# remove the uncurated_author_metadata column
patient_data = patient_data.drop('uncurated_author_metadata', axis=1)

In [ ]:
# remove the alt_sample_name column
patient_data = patient_data.drop('alt_sample_name', axis=1)

In [ ]:
# print columns with null values and their count of null values
for col in patient_data.columns:
    if patient_data[col].isnull().sum() > 0:
        print(f'{col}: {patient_data[col].isnull().sum()}')
    else:
        print(f'{col}: No missing values')

In [ ]:
# print dataset shape
patient_data.shape

In [ ]:
# view dataset
patient_data.head()

In [ ]:
# save cleaned dataset
patient_data.to_csv('Data/patient_data_cleaned.csv', index=False)

## Making Model Predictions

In [ ]:
# read cleaned dataset
patient_data = pd.read_csv('Data/patient_data_cleaned.csv')

In [ ]:
patient_data.shape

In [ ]:
patient_data.head()

In [ ]:
# function to extract patient id from the file name of the images
import re

def extract_tcga_identifier(input_string):
    # Updated regex to match up to a specific pattern, e.g., 'TCGA-XX-4746'
    match = re.search(r'(TCGA-[A-Z0-9-]+?-\d{4})', input_string)
    if match:
        return match.group(0)
    else:
        return None

In [ ]:
# extract patient id from the file name of the images of the images dataset
data_patients = [extract_tcga_identifier(file) for file in images_dataset.filenames]
print(data_patients[:5])

In [ ]:

# 'images_dataset.filenames' contains all image file names
image_files = images_dataset.filenames  # Adjust if the actual list of filenames is accessed differently

# Create a dictionary to hold patient ID to image paths mapping
patient_images = {}

# Extract patient IDs and map them to images
for filename in image_files:
    patient_id = extract_tcga_identifier(filename)
    if patient_id:  # Ensure that a patient ID was found
        full_path = os.path.join(dataset_path, filename)  # Adjust path as needed
        if patient_id in patient_images:
            patient_images[patient_id].append(full_path)
        else:
            patient_images[patient_id] = [full_path]

In [ ]:
# Example: To see the images associated with a specific patient ID
patient_id_example = 'TCGA-CM-4746'  # Replace with a real patient ID
if patient_id_example in patient_images:
    print("Images for patient ID {}: {}".format(patient_id_example, patient_images[patient_id_example]))
else:
    print("No images found for patient ID", patient_id_example)

In [ ]:
# Check some patient mappings
for pid, images in list(patient_images.items())[:10]:  # Check first 10 entries
    print("Patient ID:", pid, "has", len(images), "images.")

In [ ]:
# 'patient_data' is a pandas DataFrame that contains a column 'unique_patient_ID'
unique_patient_ids = set(patient_data['unique_patient_ID'])
print(f"Number of unique patient IDs in the patient data: {len(unique_patient_ids)}")

# Get the set of patient IDs from the image dictionary keys
image_patient_ids = set(patient_images.keys())
print(f"Number of unique patient IDs in the image data: {len(image_patient_ids)}")

# Find the intersection of both sets to see which patient IDs are present in both
matching_patient_ids = unique_patient_ids.intersection(image_patient_ids)

# Print results
print("Number of matching patient IDs:", len(matching_patient_ids))

# Number of Patient IDs with no image matches
no_image_patient_ids = unique_patient_ids.difference(image_patient_ids)
print(f"Number of patient IDs with no image matches: {len(no_image_patient_ids)}")

# Calculate the percentage of patient data IDs with images to 2 decimal places
percentage_with_images = (len(matching_patient_ids) / len(unique_patient_ids)) * 100
print(f"Percentage of patient data IDs with images: {percentage_with_images:.2f}%")

#### The same Patient has many images assciated with it 

In [ ]:
print("Class indices:", images_dataset.class_indices)

In [ ]:
def predict_image(model, image_path):
    # Load and resize the image
    image = load_img(image_path, target_size=(224, 224))  # Adjust target_size if different

    # Convert the image pixels to a numpy array
    image_array = img_to_array(image)

    # Preprocess the image for the model
    image_preprocessed = preprocess_input(image_array)

    # Expand the dimensions to fit the model input format, assumes model expects batches
    image_batch = np.expand_dims(image_preprocessed, axis=0)

    # Make a prediction
    prediction = model.predict(image_batch)
    return prediction[0][0]

In [ ]:
# test prediction function on a single image
prediction = predict_image('Data/EXTRA/ALL_IMAGES/MSI/blk-AAADECQEWVSD-TCGA-CM-4746-01Z-00-DX1.png')
print(prediction)

In [ ]:
# Filter patient IDs that exist in patient_data
valid_patient_ids = set(patient_data['unique_patient_ID'])

# Filter the dictionary to include only valid patient IDs
filtered_patient_images_dict = {k: v for k, v in patient_images.items() if k in valid_patient_ids}

# print number of valid patient IDs
print("Number of valid patient IDs:", len(filtered_patient_images_dict))

In [ ]:
patient_predictions = {}

for patient_id, filenames in filtered_patient_images_dict.items():
    patient_probs = []
    for filename in filenames:
        prob = predict_image(model, filename)
        patient_probs.append(prob)
    
    # Store the average probability for the patient
    if patient_probs:
        patient_predictions[patient_id] = np.mean(patient_probs)
    else:
        patient_predictions[patient_id] = None  # Handle case where no images are available for a valid ID

# This will give you a dictionary with each patient ID and their average prediction across all their images.

In [ ]:
# Print the first few patient predictions
for patient_id, avg_prob in list(patient_predictions.items())[:5]:
    print("Patient ID:", patient_id, "Average Probability:", avg_prob)

In [ ]:
# Convert to DataFrame
predictions_df = pd.DataFrame(list(patient_predictions.items()), columns=['unique_patient_ID', 'average_prediction'])

# Merge with patient data
patient_data_predictions = pd.merge(patient_data, predictions_df, on='unique_patient_ID', how='left')

In [ ]:
# add predicted_label column to patient_data_predictions (<0.5: MSI, >=0.5: MSS)
# Assuming 'average_prediction' is the column with the prediction probabilities
patient_data_predictions['predicted_label'] = np.where(
    patient_data_predictions['predicted_probability'].isna(), 
    'Unknown',  # Assign 'Unknown' where the prediction is NaN
    np.where(patient_data_predictions['predicted_probability'] < 0.5, 'MSI', 'MSS')
)

In [ ]:
# add actual_label column to patient_data_predictions 

# Extract actual labels from the first image path or assign 'Unknown' if no valid match exists
actual_labels = {}
for patient_id in patient_data_predictions['unique_patient_ID']:  # Loop through expected patient IDs
    filepaths = filtered_patient_images_dict.get(patient_id)  # Safely get the list of file paths
    if filepaths:  # Make sure there is at least one image
        # Check if 'MSS' or 'MSI' is in the path of the first image
        actual_label = 'MSS' if 'MSS' in filepaths[0] else 'MSI'
    else:
        actual_label = 'Unknown'  # Assign 'Unknown' if no images or not a valid patient ID match
    actual_labels[patient_id] = actual_label

# Convert the dictionary to a DataFrame
labels_df = pd.DataFrame(list(actual_labels.items()), columns=['unique_patient_ID', 'actual_label'])

# Merge this with your existing patient predictions DataFrame
patient_data_predictions = pd.merge(patient_data_predictions, labels_df, on='unique_patient_ID', how='left')

In [ ]:
patient_data_predictions.head(10)

In [ ]:
patient_data_predictions.shape

In [ ]:
# drop rows where actual_label or predicted_label is unknown
patient_data_predictions = patient_data_predictions[~patient_data_predictions['actual_label'].isin(['Unknown'])]

In [ ]:
patient_data_predictions.shape

In [ ]:
# print accuracy by comparing actual_label and prediction_label to 2 decimal places as a percentage
accuracy = (patient_data_predictions['actual_label'] == patient_data_predictions['predicted_label']).mean() * 100
print(f"Accuracy: {accuracy:.2f}%")

In [ ]:
# save as predictions_patient_data.csv
patient_data_predictions.to_csv('Data/patient_data_final.csv', index=False)

In [ ]:
# print the total number of images used for prediction 
# Count the total number of images that matched with a patient ID
total_matched_images = sum(len(filepaths) for filepaths in filtered_patient_images_dict.values())

print("Total number of images matched with patients:", total_matched_images)